In [ ]:
import os

# 第一步：定义你想要的路径
my_path = '/home/haod6/hf_cache'

# 第二步：务必先创建这个文件夹，否则检查空间会报错
if not os.path.exists(my_path):
    os.makedirs(my_path)
    print(f"✅ 文件夹已创建: {my_path}")

# 第三步：先设置环境变量，再 import huggingface 相关的库
os.environ['HF_HOME'] = my_path

In [ ]:
from huggingface_hub import constants
print(f"--- 确认结果 ---")
print(f"现在的缓存地址: {constants.HF_HOME}")

# 检查空间
stat = os.statvfs(my_path)
free_gb = (stat.f_bavail * stat.f_frsize) / (1024**3)
print(f"该路径剩余可用空间: {free_gb:.2f} GB")

In [ ]:
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from torch.utils.data import DataLoader
from sft import DataCollator, SFTData
import sys
from tqdm import tqdm
# from .autonotebook import tqdm as notebook_tqdm
import json
import shutil
import argparse
import random
from vllm import LLM, SamplingParams
import torch.nn.functional as F

from sampling import sampling, GRPODataset, make_prompt, extract_gt_answer, extract_answer_from_model, clean_math_answer, check_format, compute_total_reward
import gc
import time
import re

from drgrpo_grader import grade

In [ ]:
if __name__ == "__main__":

    os.environ["HF_HOME"] = "/home/haod6/hf_cache"
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    
    # MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B"
    # MODEL_PATH = "Qwen/Qwen2.5-Math-1.5B"
    MODEL_PATH = "/home/haod6/grpo_16_round_11/grpo_step_125"


    BATCH_SIZE = 4
    lr = 1e-6
    EPOCH = 2

    if not torch.cuda.is_available():
        print("❌ Error: CUDA is not available")
        sys.exit(1)

    # Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    data_collator = DataCollator(tokenizer)

    # Load Model
    # device = torch.device("cuda")
    device = torch.device("cuda:0") 
    SAVE_PATH = f"/home/haod6/grpo_last"
    os.makedirs(SAVE_PATH, exist_ok=True)

    # path = f"/work/nvme/bfdu/mdong1/assignment5/data/math/subsets/sample_{args.sample_size}/train.jsonl"
    path = f"/home/haod6/data/math/train.jsonl"
    if not os.path.exists(path):
        print(f"Error: Data file not found at {path}")
    else:
        with open(path, 'r') as f:
            train_data = [json.loads(line) for line in f]
        f.close()


    val_path =  "/home/haod6/data/math/val.jsonl"
    with open(val_path, 'r') as f:
        val_data = [json.loads(line) for line in f]
    # val_dataset = SFTData(val_data, tokenizer)
    # val_loader = DataLoader(
    #     val_dataset, 
    #     batch_size=BATCH_SIZE,      
    #     shuffle=True,     
    #     collate_fn=data_collator 
    # )

    n_prompts = 32          # 每次抽 32 个不同问题
    group_size = 16         # 每个问题 8 个答案 (Total = 256 samples)
    micro_batch_size = 2    # 显存能跑的微批次
    grad_accum_steps = 128  # 256 / 2 = 128 步累积，等效于一个大 batch 更新
    n_grpo_steps = 200
    epsilon = 0.2
    test_step = 5
    eval_step = 5

    all_losses = []
    all_accs = []
    all_val_losses = []
    all_val_steps = []

    best_val_loss = 0.0
    global_step = 0

    for grpo_step in range(n_grpo_steps):
        
        # 随机选 32 个问题
        D_d = random.sample(train_data, n_prompts)

        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        llm = LLM(model=MODEL_PATH, max_model_len=2048, gpu_memory_utilization=0.7, compilation_config={"level": 0}, tensor_parallel_size=1,)


        G_traj = sampling(D_d=D_d, llm=llm, n_G=group_size)

        total_reward_sum = sum(t['reward'] for t in G_traj)
        format_ok_sum = sum(1 for t in G_traj if t['format_reward'] > 0)
        answer_ok_sum = sum(1 for t in G_traj if t['answer_reward'] > 0)
        
        avg_reward = total_reward_sum / len(G_traj)
        format_rate = format_ok_sum / len(G_traj)
        answer_acc = answer_ok_sum / len(G_traj)
        
        del llm 
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        # time.sleep(2)

        os.environ["CUDA_VISIBLE_DEVICES"] = "0" 

        policy_model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            torch_dtype=torch.bfloat16, 
            trust_remote_code=True
        )
        policy_model.to(device)
        policy_model.gradient_checkpointing_enable()
        # optimizer = torch.optim.AdamW(policy_model.parameters(), lr=lr, weight_decay=0.0, betas=(0.9, 0.95),)
        optimizer = torch.optim.AdamW(
            policy_model.parameters(), 
            lr=1e-6,           # 从 5e-6 降到 1e-6
            weight_decay=0.01, # 增加微弱的权重衰减
            betas=(0.9, 0.99), # Beta2 调高一点，增加动量平滑度
            eps=1e-8
        )

        grpo_dataset = GRPODataset(G_traj, tokenizer)
        loader = DataLoader(
            grpo_dataset, 
            batch_size=4,      
            shuffle=True,     
            collate_fn=data_collator 
        )
        
        optimizer.zero_grad()
        policy_model.train()

        epochs_per_rollout_batch = 1
        for epoch in range(epochs_per_rollout_batch):
            for step, batch in tqdm(enumerate(loader), total=len(loader), desc=f"GRPO Step [{grpo_step+1} /{n_grpo_steps}] | Epoch [{epoch+1}/{epochs_per_rollout_batch}]"):

                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                advantages = batch['advantage'].to(device)
                old_log_probs = batch['old_log_probs'].to(device)

                # 前向传播
                outputs = policy_model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits

                # 错位对齐计算 Log Probs
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = labels[..., 1:].contiguous()
                
                log_probs_dist = F.log_softmax(shift_logits, dim=-1)
                valid_labels = shift_labels.clamp(min=0)
                token_log_probs = torch.gather(log_probs_dist, dim=-1, index=valid_labels.unsqueeze(-1)).squeeze(-1)

                loss_mask = (shift_labels != -100).to(logits.dtype)

                # 计算重要性采样比率 Ratio
                current_old_lps = old_log_probs[:, :token_log_probs.size(1)]

                ratio = torch.exp(token_log_probs - current_old_lps)
                adv = advantages.unsqueeze(1) # [batch, 1] 广播到序列长度

                # GRPO Clip Loss
                surr1 = ratio * adv
                surr2 = torch.clamp(ratio, 1.0 - epsilon, 1.0 + epsilon) * adv
                token_loss = -torch.min(surr1, surr2)

                # ================= 新增：KL 散度惩罚 =================
                approx_kl = 0.5 * ((token_log_probs - current_old_lps) ** 2)
                
                # kl_coeff (Beta 系数) 决定了惩罚力度。
                # 推荐值：0.01 ~ 0.05 之间。如果你发现输出乱码，就调大；如果发现模型学不动，就调小。
                kl_coeff = 0.02 
                
                # 将 KL 惩罚加到损失中 (由于 token_loss 本身是我们要 minimize 的负回报，所以 KL 惩罚是加号)
                token_loss = token_loss + kl_coeff * approx_kl
                # ================= 新增：KL 散度惩罚 =================
                
                # 掩码聚合损失
                loss = (token_loss * loss_mask).sum() / loss_mask.sum()

                # --- 重点：梯度累积逻辑 ---
                # 损失除以累积步数
                (loss / grad_accum_steps).backward()

                if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(loader):
                    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
                    optimizer.step()
                    optimizer.zero_grad()

                with torch.no_grad():
                    # 准确率计算
                    preds = shift_logits.argmax(dim=-1)
                    correct_mask = (shift_labels != -100)
                    acc = (preds == shift_labels)[correct_mask].float().mean() if correct_mask.sum() > 0 else torch.tensor(0.0)
                    
                    # 熵计算 (用于监控策略是否坍缩)
                    entropy = -(torch.exp(log_probs_dist) * log_probs_dist).sum(dim=-1)
                    avg_entropy = (entropy * loss_mask).sum() / loss_mask.sum()

                curr_loss = loss.item()
                curr_acc = acc.item()
                all_losses.append(curr_loss)
                all_accs.append(curr_acc)

        if grpo_step % test_step == 0:
            tqdm.write(f"GRPO round {grpo_step+1}: Neg-Log Loss: {all_losses[-1]} | Acc: {all_accs[-1]}")

        # if (grpo_step + 1) % eval_step == 0:
        #     policy_model.eval()
        #     val_loss = 0
        #     val_acc = 0
        #     max_eval_batches = 50 
            
        #     tqdm.write(f"\n🔍 Running Evaluation for Round {grpo_step+1}...")
            
        #     with torch.no_grad():
        #         for v_step, val_batch in enumerate(val_loader):
        #             if v_step >= max_eval_batches: break
                    
        #             v_input_ids = val_batch['input_ids'].to(device)
        #             v_attention_mask = val_batch['attention_mask'].to(device)
        #             v_labels = val_batch['labels'].to(device)
                    
        #             v_outputs = policy_model(
        #                 input_ids=v_input_ids,
        #                 attention_mask=v_attention_mask,
        #                 labels=v_labels
        #             )
                    
        #             val_loss += v_outputs.loss.item()
        #             v_logits = v_outputs.logits
        #             v_preds = v_logits[..., :-1, :].argmax(dim=-1)
        #             v_shift_labels = v_labels[..., 1:]
        #             v_correct = (v_preds == v_shift_labels) & (v_shift_labels != -100)
        #             val_acc += v_correct.float().sum() / (v_shift_labels != -100).float().sum()

        #     avg_val_loss = val_loss / min(len(val_loader), max_eval_batches)
        #     avg_val_acc = val_acc / min(len(val_loader), max_eval_batches)
            
        #     tqdm.write(f"🌍 Round {grpo_step+1} | Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_acc:.4f}")

        #     all_val_losses.append(avg_val_loss)
        #     all_val_steps.append(grpo_step + 1)

        #     if avg_val_loss < best_val_loss:
        #         best_val_loss = avg_val_loss
        #         best_model_path = os.path.join(SAVE_PATH, "best_model")
        #         policy_model.save_pretrained(best_model_path)
        #         tokenizer.save_pretrained(best_model_path)
        #         tqdm.write(f"⭐ New Best Model Saved (Loss: {best_val_loss:.4f})")

        latest_path = os.path.join(SAVE_PATH, "latest")
        policy_model.save_pretrained(latest_path)
        tokenizer.save_pretrained(latest_path)
        MODEL_PATH = latest_path
        
        if (grpo_step + 1)% eval_step == 0:
            snapshot_path = os.path.join(SAVE_PATH, f"grpo_step_{grpo_step+1}")
            policy_model.save_pretrained(snapshot_path)
            tokenizer.save_pretrained(snapshot_path)
            
        del policy_model, optimizer, loader, grpo_dataset, G_traj
        gc.collect(); torch.cuda.empty_cache();torch.cuda.synchronize()

        tqdm.write(f"✅ Round {grpo_step+1} Completed. Weights updated for next sampling.")
        # tqdm.write(f"Allocated: {torch.cuda.memory_allocated(1)/1024**3:.1f} GB")  # 真正占用
        # tqdm.write(f"Reserved:  {torch.cuda.memory_reserved(1)/1024**3:.1f} GB")

        if (grpo_step + 1) % eval_step == 0:
            tqdm.write(f"\n🔍 Running vLLM Evaluation for Round {grpo_step+1}...")

            eval_llm = LLM(
                model=MODEL_PATH,
                max_model_len=2048,
                gpu_memory_utilization=0.7,
                tensor_parallel_size=1,
            )

            eval_samples = random.sample(val_data, min(64, len(val_data)))
            eval_prompts = [make_prompt(item['problem']) for item in eval_samples]
            
            # 获取验证集标准答案并清洗
            # eval_gt_answers = [clean_math_answer(extract_gt_answer(item['solution'])) for item in eval_samples]
            eval_gt_answers = [item['solution'] for item in eval_samples]

            eval_sampling_params = SamplingParams(
                n=1,
                temperature=0.0, # 验证时通常用 0 提高确定性
                max_tokens=1024,
                stop=["</answer>"],
                include_stop_str_in_output=True,
            )
            eval_outputs = eval_llm.generate(eval_prompts, eval_sampling_params)

            # --- 核心修正：统计验证集表现 ---
            val_correct = 0
            val_format_ok = 0

            for i, output in enumerate(eval_outputs):
                generated_text = output.outputs[0].text
                gt_ans = eval_gt_answers[i]
                
                # 2. 检查格式
                is_fmt_ok = "</think>" in generated_text and "<answer>" in generated_text
                if is_fmt_ok:
                    val_format_ok += 1
                
                match = re.search(r"<answer>(.*?)(?:</answer>|$)", generated_text, re.DOTALL)
                if match:
                    ans_content = match.group(1).strip()
                else:
                    ans_content = ""

                # 4. 用 math_grader 判分（引入新逻辑）
                # 注意：确保在文件顶部加上 from math_grader import grade
                is_correct = grade(ans_content, gt_ans, fast=False) # 验证集可以用 fast=False 进行最严谨的评测
                
                if is_correct:
                    val_correct += 1
                    
            for i, output in enumerate(eval_outputs[:6]):
                generated_text = output.outputs[0].text
                gt_ans = eval_gt_answers[i]
                
                # 2. 检查格式
                is_fmt_ok = "</think>" in generated_text and "<answer>" in generated_text
                match = re.search(r"<answer>(.*?)(?:</answer>|$)", generated_text, re.DOTALL)
                if match:
                    ans_content = match.group(1).strip()
                else:
                    ans_content = ""

                is_correct = grade(ans_content, gt_ans, fast=False)
                
                if i < 5:
                    tqdm.write(f"📖 Printing output...")
                    tqdm.write(f"Format Match: {is_fmt_ok} | Ans Reward: {is_correct}")
                    tqdm.write(f"Cleaned Ans: {ans_content} | Ground Truth: {extract_gt_answer(gt_ans)}")

            avg_val_acc = val_correct / len(eval_samples)
            avg_val_format = val_format_ok / len(eval_samples)

            tqdm.write(f"🌍 Round {grpo_step+1} | Val Answer Acc: {avg_val_acc:.4f} | Val Format Rate: {avg_val_format:.4f}")

            # 根据验证集表现保存最优模型
            if avg_val_acc >= best_val_loss: # 借用原变量名·
                best_val_loss = avg_val_acc
                best_model_path = os.path.join(SAVE_PATH, "best_model")
                import shutil
                if os.path.exists(best_model_path): shutil.rmtree(best_model_path)
                shutil.copytree(latest_path, best_model_path)
                tqdm.write(f"⭐ New Best Model Saved (Val Acc: {best_val_loss:.4f})")

            del eval_llm
            gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()



    metrics_data = {
        "model_name": MODEL_PATH,
        "total_steps": len(all_losses),
        "losses": all_losses,
        "accuracies": all_accs,
        "eval": {"steps": all_val_steps, "losses": all_val_losses}
    }
    with open(os.path.join(SAVE_PATH, "train_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics_data, f, indent=4)